In [1]:
# =============================================================================
# Libaries
# =============================================================================
#UI
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

#Data handling
import pandas as pd
import re
import xml.etree.ElementTree as ET
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import openpyxl
import io
import os
import numpy as np
import shutil

# =============================================================================
# Configurations
# =============================================================================
excel_output_file = "ReviewOutput.xlsx"
header = ["ID","PMID", "FIRST_AUTHOR", "TITLE", "ABSTRACT", "YEAR", "TYPE", "DOI", "URL", "DATABASE"]
column_widths = {"A": 10, "B": 15, "C": 25, "D": 40, "E": 40, "F": 10, "G": 15, "H": 20, "I": 30, "J": 15}


# =============================================================================
# Functions
# =============================================================================
def display_main_menu():
    clear_output(wait=True)
    display(HTML("<i>Review Helper</i>"))
    database_uploader_button = widgets.Button(description="Database Uploader")
    screening_button = widgets.Button(description="Screening")
    compare_button = widgets.Button(description="Compare Decisions")
    extraction_button = widgets.Button(description="Data Extraction")

    database_uploader_button.on_click(on_database_uploader_clicked)
    screening_button.on_click(on_screening_clicked)
    compare_button.on_click(on_compare_clicked)
    extraction_button.on_click(on_extraction_clicked)
    display(widgets.HBox([database_uploader_button, screening_button, compare_button, extraction_button]))


def prepare_worksheet(worksheet):
    for idx, col_name in enumerate(header, start=1):
        cell = worksheet.cell(row=1, column=idx, value=col_name) #Writing the header
        cell.font = Font(bold=True); #Bolding the header cells
        cell.alignment = Alignment(horizontal="center", vertical="center") #Centering the header cells

    for i, col_letter in enumerate(column_widths.keys()):
        worksheet.column_dimensions[col_letter].width = column_widths[col_letter] #Applying the column widths
    worksheet.row_dimensions[1].height = 20 #Setting the column heights of the header row


def parse_pubmed_text_data(text):
    records = re.split(r"\n(?=PMID- )", text.strip()) #New article starts at PMID
    articles = []

    for article_id, record in enumerate(records, start=1): #Every article gets an id starting from 1
        fields = {} #Tag-text
        current_tag = None

        for line in record.splitlines():
            if len(line) >= 6 and line[4:6] == "- ": #4:6 beacuse EDAT- so 3:6 with " - " would be good
                current_tag = line[:4].strip() #Tags are four charater long
                value = line[6:].strip()
                fields.setdefault(current_tag, []).append(value)
            elif current_tag and line.startswith("      "): #If the text is not in only one line
                fields[current_tag][-1] += " " + line.strip()

        publication_date = fields.get("DP", [""])[0]
        year = int(publication_date[:4]) if publication_date[:4].isdigit() else None

        doi = ""
        identifier_values = fields.get("LID", []) + fields.get("AID", []) #list of the possible doi places

        for value in identifier_values: # First look for identifiers explicitly marked as DOI
            if value.lower().endswith("[doi]"):
                doi = re.sub(r"\s*\[doi\]\s*$", "", value, flags=re.IGNORECASE).strip()
                break

        if not doi: # If there is no [doi] tag, check whether the identifier looks like a DOI
            for value in identifier_values:
                value = value.strip()
                if re.search(r"\[[^\]]+\]\s*$", value): # Ignore identifiers explicitly marked as something else, e.g. [pii]
                    continue
                if re.fullmatch(r"10\.\d{4,9}/\S+", value, re.IGNORECASE): #doi format
                    doi = value
                    break

        pmid = fields.get("PMID", [""])[0]

        articles.append({
            "ID": article_id,
            "PMID": pmid,
            "FIRST_AUTHOR": fields.get("FAU", [""])[0],
            "TITLE": " ".join(fields.get("TI", [])),
            "ABSTRACT": " ".join(fields.get("AB", [])),
            "YEAR": year,
            "TYPE": fields.get("PT", [])[0],
            "DOI": doi,
            "URL": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "DATABASE": "PubMed"
        })
    return pd.DataFrame(articles)

def parse_arxiv_xml_data(xml_data_text):
    root = ET.fromstring(xml_data_text)
    namespaces = {"atom": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom"} #arXiv xml uses this namespace
    articles = []

    for article_id, entry in enumerate(root.findall("atom:entry", namespaces), start=1):
        title = entry.findtext("atom:title", default="", namespaces=namespaces).strip()
        abstract = entry.findtext("atom:summary", default="", namespaces=namespaces).strip()
        published = entry.findtext("atom:published", default="", namespaces=namespaces).strip()
        link = entry.findtext("atom:id", default="", namespaces=namespaces).strip()
        doi = entry.findtext("arxiv:doi", default="", namespaces=namespaces).strip()

        first_author = ""
        first_author_element = entry.find("atom:author", namespaces)
        if first_author_element is not None:
            first_author = first_author_element.findtext("atom:name", default="", namespaces=namespaces).strip()

        articles.append({
            "ID": article_id,
            "PMID": "", #Empty for arXiv articles
            "FIRST_AUTHOR": first_author,
            "TITLE": title,
            "ABSTRACT": abstract,
            "YEAR": int(published[:4]) if published else None,
            "TYPE": "preprint" if not doi else "published",
            "DOI": doi,
            "URL": link,
            "DATABASE": "arXiv"
        })
    return pd.DataFrame(articles)


def parse_ris_data(text):
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    records = re.split(r"\n(?=TY  -)", text.strip()) #New article starts at TY
    articles = []

    for article_id, record in enumerate(records, start=1): #Every article gets an id starting from 1
        fields = {} #tag-text
        current_tag = None

        for line in record.splitlines():
            if len(line) >= 6 and line[4:6] == "- ": #4:6 beacuse EDAT- so 3:6 with " - " would be good
                current_tag = line[:4].strip() #Tags are four charater long
                value = line[6:].strip()
                fields.setdefault(current_tag, []).append(value)
            elif current_tag and line.startswith("      "):
                fields[current_tag][-1] += " " + line.strip()

        doi = " ".join(fields.get("DO", []))

        if " ".join(fields.get("DB", [])) == "Scopus": #Checking for DB tag for database selection
          database = "Scopus"
          url = " ".join(fields.get("UR", []))
        else:
          database = "IEEE Xplore"
          url = f"https://ieeexplore.ieee.org/document/{doi.split(".")[-1]}"

        articles.append({
            "ID": article_id,
            "PMID": "",
            "FIRST_AUTHOR": fields.get("AU", [""])[0],
            "TITLE": " ".join(fields.get("TI", [])),
            "ABSTRACT": " ".join(fields.get("AB", [])),
            "YEAR": " ".join(fields.get("PY", [])),
            "TYPE": " ".join(fields.get("TY", [])).removeprefix("label.ris.referenceType."),
            "DOI": doi,
            "URL": url,
            "DATABASE": database
        })
    return pd.DataFrame(articles)


def parse_text_data(text_content, file_extension):
    if "PMID- " in text_content: #PubMed
        return parse_pubmed_text_data(text_content)
    elif "<id>" in text_content and "<feed" in text_content: #arXiv
        return parse_arxiv_xml_data(text_content)
    elif file_extension == "ris": #IEEE and Scopus
        return parse_ris_data(text_content)
    else:
        raise ValueError("Unsupported file format. Please upload a correct file.")


def export_dataframes_to_excel(all_dfs, db_names, output_file_path, header_cols, worksheet_prep_func):
    wb = Workbook()
    if "Sheet" in wb.sheetnames:
        wb.remove(wb["Sheet"])

    for df_idx, df_to_save in enumerate(all_dfs):
        sheet_name = db_names[df_idx][:31].replace('/', '_').replace('\\', '_').replace('?', '_').replace('*', '_').replace('[', '_').replace(']', '_').replace(':', '_')
        ws = wb.create_sheet(title=sheet_name)
        worksheet_prep_func(ws)

        df_display = df_to_save.rename(columns={'PUBMED_URL': 'URL', 'ARXIV_URL': 'URL'}).reindex(columns=header_cols)

        for r_idx, row in enumerate(df_display.itertuples(index=False), start=2):
            ws.append(row)

        for r_idx, article in enumerate(df_to_save.to_dict(orient='records'), start=2):
            current_row = r_idx

            if article.get("DOI") and article["DOI"] != 'NO_DOI':
                doi_col_idx = None
                for col_idx, col_name in enumerate(header_cols, 1):
                    if col_name == "DOI":
                        doi_col_idx = col_idx
                        break
                if doi_col_idx is not None:
                    doi_cell = ws.cell(row=current_row, column=doi_col_idx)
                    doi_cell.value = article["DOI"]
                    doi_cell.hyperlink = f"https://doi.org/{article['DOI']}"
                    doi_cell.style = "Hyperlink"

            url_col_index = None
            for col_idx, col_name in enumerate(header_cols, 1):
                if col_name == "URL":
                    url_col_index = col_idx
                    break
            if article.get("URL") and article["URL"] != 'NO_URL' and url_col_index is not None:
                url_cell = ws.cell(row=current_row, column=url_col_index)
                if article.get("DATABASE") == "PubMed":
                    url_cell.value = "PubMed Link"
                elif article.get("DATABASE") == "arXiv":
                    url_cell.value = "arXiv Link"
                else:
                    url_cell.value = "Link"
                url_cell.hyperlink = article["URL"]
                url_cell.style = "Hyperlink"
    wb.save(output_file_path)


def deduplicate_all_dfs(all_dfs, db_names):
    processed_dfs = []
    records_by_database = {}

    for df in all_dfs:
        df = df.copy()
        database_name = df["DATABASE"].iloc[0]
        processed_dfs.append(df)
        records_by_database[database_name] = len(df)

    merged_df = pd.concat(processed_dfs, ignore_index=True)
    merged_df['ID'] = range(1, len(merged_df) + 1)
    merged_df['TITLE_CLEAN'] = merged_df['TITLE'].astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True).str.rstrip('.!?')
    merged_df['YEAR_CLEAN'] = merged_df['YEAR'].astype(str).str.strip()

    # base values for the detection
    merged_df["DUPLICATION"] = "N"
    merged_df["DUPLICATION_METHOD"] = ""
    merged_df["DUPLICATE_OF_ID"] = ""

    ##### DOI EXACT MATCHING ####
    doi_duplicate = (merged_df["DOI"].ne("") & merged_df["DOI"].duplicated(keep="first")) #Checking for not empty, duplicated dois --> True: duplication, False: not duplication
    first_doi_ids = (merged_df[merged_df["DOI"].ne("")].drop_duplicates(subset="DOI", keep="first").set_index("DOI")["ID"]) #First occurrence ID for every DOI
    merged_df.loc[doi_duplicate, "DUPLICATION"] = "Y" #chancing the values for duplications
    merged_df.loc[doi_duplicate, "DUPLICATION_METHOD"] = "doi_exact" #setting the method for doi
    merged_df.loc[doi_duplicate, "DUPLICATE_OF_ID"] = (merged_df.loc[doi_duplicate,"DOI"].map(first_doi_ids))

    #### TITLE EXACT MATCHING ####
    title_duplicate = (merged_df["TITLE_CLEAN"].ne("") & merged_df["TITLE_CLEAN"].duplicated(keep="first"))
    first_title_ids = (merged_df[merged_df["TITLE_CLEAN"].ne("")].drop_duplicates(subset="TITLE_CLEAN", keep="first").set_index("TITLE_CLEAN")["ID"])
    title_duplicate_mask = (title_duplicate & merged_df["DUPLICATION"].eq("N")) # Only mark records that have not already been identified
    merged_df.loc[title_duplicate_mask, "DUPLICATION"] = "S"
    merged_df.loc[title_duplicate_mask, "DUPLICATION_METHOD"] = "title_exact"
    merged_df.loc[title_duplicate_mask, "DUPLICATE_OF_ID"] = (merged_df.loc[title_duplicate_mask, "TITLE_CLEAN"].map(first_title_ids))

    merged_df = merged_df.drop(columns=["TITLE_CLEAN", "YEAR_CLEAN"]) #Deleting the preprocessing columns

    stats = {
        "total_records": len(merged_df),
        "records_by_database": records_by_database,
        "duplicated_records": int((merged_df["DUPLICATION"] == "Y").sum()),
        "suspected_duplicates": int((merged_df["DUPLICATION"] == "S").sum())
    }
    return merged_df, stats


def final_df_construct(display_df, inclusion_decisions, full_text_decisions):
    display_df['Title_Abstract_Include'] = display_df['ID'].apply(lambda x: {'Include': 'Y', 'Exclude': 'N', 'Exclude-Duplication': 'N'}.get(inclusion_decisions.get(x, 'Pending'), 'Pending'))

    def get_full_text_decision_status(article_id):
        decision = full_text_decisions.get(article_id)
        if decision == 'Include':
            return 'Y'
        if decision and decision.startswith('Exclude'): #Covers 'Exclude', 'Exclude - No Full Text', etc.
            return 'N'
        return 'Pending'
    display_df['Full_Text_Include'] = display_df['ID'].apply(get_full_text_decision_status)

    def get_full_text_exclusion_reason(article_id):
        decision = full_text_decisions.get(article_id)
        if decision and decision.startswith('Exclude - '):
            return decision.replace('Exclude - ', '')
        else:
            return ''
    display_df['Full_Text_Exclusion_Reason'] = display_df['ID'].apply(get_full_text_exclusion_reason)

    if 'PMID' in display_df.columns: #Remove 'PMID' column if it exists
        display_df = display_df.drop(columns=['PMID'])
    return display_df


def saving_menu(df, inclusion_decisions, full_text_decisions, dynamic_output_area, stage):
    with dynamic_output_area:
         clear_output(wait=True)
         display(HTML("<b>Final DataFrame</b>"))
         save_final_button = widgets.Button(description="Save Final")
         save_final_output = widgets.Output()
         back_to_main_button = widgets.Button(description="Back to Main Menu", layout=widgets.Layout(visibility='hidden'))

         if stage in ("DB_Upload", "Screening"):
              display_df = final_df_construct(df, inclusion_decisions, full_text_decisions)
         else:
              display_df = df

         def on_back_to_main_clicked(b):
             clear_output()
             display_main_menu()
         back_to_main_button.on_click(on_back_to_main_clicked)

         def on_save_final_clicked(b_final_save):
             nonlocal summary_html
             with save_final_output:
                 clear_output()
                 try:
                     with pd.ExcelWriter(excel_output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                         display_df.to_excel(writer, sheet_name='Final', index=False)

                     wb = writer.book
                     ws_final = wb['Final']

                     for r_idx, row_data in enumerate(display_df.to_dict(orient='records'), start=2): #Hyperlinks
                         if row_data.get("DOI"):
                             doi_col_idx = display_df.columns.get_loc("DOI") + 1
                             doi_cell = ws_final.cell(row=r_idx, column=doi_col_idx)
                             doi_cell.hyperlink = f"https://doi.org/{row_data['DOI']}"
                             doi_cell.style = "Hyperlink"

                         if row_data.get("URL"):
                             url_col_idx = display_df.columns.get_loc("URL") + 1
                             url_cell = ws_final.cell(row=r_idx, column=url_col_idx)
                             url_cell.hyperlink = row_data["URL"]
                             url_cell.style = "Hyperlink"
                     wb.save(excel_output_file) #Saving the workbook again to apply hyperlink changes
                     back_to_main_button.layout.visibility = 'visible' #Back button visible after saving
                 except Exception as e:
                     display(HTML(f"<b style='color:red;'>Error saving Final Review: {e}</b>"))
         save_final_button.on_click(on_save_final_clicked)
         display(widgets.HBox([save_final_button, back_to_main_button, save_final_output]))

         if stage == "Screening":
            automatic_duplicates = (display_df["DUPLICATION"].eq("Y").sum())
            manual_duplicates = (display_df["Full_Text_Exclusion_Reason"].eq("Duplicate")& display_df["DUPLICATION"].eq("N")).sum()
            after_dedup = len(display_df) - (automatic_duplicates + manual_duplicates)

            title_abstract_excluded = (display_df["Title_Abstract_Include"].eq("N").sum())
            after_title_abstract = after_dedup - title_abstract_excluded
            full_text_excluded = (display_df["Full_Text_Include"].eq("N")& display_df["Title_Abstract_Include"].eq("Y")).sum()

            summary = [
                f"<i>Records screened:</i> {display_df["Title_Abstract_Include"].ne("Pending").sum()} / {len(display_df)}",
                "-------------------------------",
                f"<i>Auto duplicates:</i> {automatic_duplicates}",
                f"<i>Manual duplicates:</i> {manual_duplicates}",
                f"<i>Total duplicates:</i> {automatic_duplicates + manual_duplicates}",
                f"<i>Remaining after de-duplication:</i> {after_dedup}",
                "-------------------------------",
                f"<i>Excluded at title/abstract:</i> {title_abstract_excluded}",
                f"<i>Remaining after title/abstract:</i> {after_title_abstract}",
                f"<i>Excluded at full text:</i> {full_text_excluded}",
                f"<i>Remaining after full text:</i> {after_title_abstract - full_text_excluded}"
            ]

            summary_html = f"""<div style="display:
                  inline-block; width: fit-content; max-width: 100%;
                  border: 1px solid #ddd; padding: 10px; margin: 10px 0; border-radius: 6px;
                  background-color: #f9f9f9; line-height: 1.6;">
                  {"<br>".join(summary)}
            </div>"""
            display(HTML(summary_html))
         display(display_df)


def excel_uploader(file_upload, uploaded_file_name):
    global excel_output_file #To modify the excel name
    uploaded_file_content = list(file_upload.value.values())[0]['content']

    save_path = os.path.join(os.getcwd(), uploaded_file_name) #Colab's file system
    with open(save_path, 'wb') as f:
        f.write(uploaded_file_content)

    excel_file = pd.ExcelFile(save_path)
    if "Final" in excel_file.sheet_names:
        with open(save_path, 'wb') as f:
             f.write(uploaded_file_content)

        excel_output_file = uploaded_file_name
        df = pd.read_excel(io.BytesIO(uploaded_file_content), sheet_name="Final") #Saving the Final to a dataframe
        df = df.fillna(" ")
        return df
    else:
        return None


def title_abstract_inclusion(df, inclusion_decisions, full_text_decisions, dynamic_output_area):
    with dynamic_output_area:
        clear_output(wait=True)
        display(HTML("<b>Title-Abstract Screening</b>"))
        id_input = widgets.Text(description="Article ID:", placeholder="Enter Article ID (e.g., 1, 2, ...)")
        font_size_slider = widgets.IntSlider(value=15, min=10, max=24, tep=1, description='Font Size:', continuous_update=True)

        title_display_widget = widgets.HTML(value="<p><b>Title:</b></p>")
        abstract_display_widget = widgets.HTML(value="<p><b>Abstract:</b></p>")
        include_button = widgets.Button(description="Include")
        exclude_button = widgets.Button(description="Exclude")
        duplication_button = widgets.Button(description="Duplicate")
        exclude_duplicate_button = widgets.Button(description="Exclude Duplicate")
        current_decision_label = widgets.Label(value="Current Decision: Pending")
        buttons_container = widgets.HBox() #For the buttons

        log_output = widgets.Output()
        current_displayed_article_info = {'article_id': None}

        def make_decision(decision): #Decision to include or exclude the article
             nonlocal inclusion_decisions
             article_id = current_displayed_article_info['article_id']

             if article_id is not None:
                 article = df[df['ID'] == article_id].iloc[0]

                 inclusion_decisions[article_id] = decision #Updatig the current decision
                 current_decision_label.value = f"Current Decision: {inclusion_decisions.get(article_id, 'Pending')}"

                 if decision == 'Include':
                    if full_text_decisions.get(article_id) in ('Exclude - Title-Abstract', 'Exclude - Duplicate'):
                        full_text_decisions[article_id] = 'Pending'
                        with log_output:
                            print(f"Full Text decision for Article {article_id} reset to Pending.")

                 if article['DUPLICATION'] == 'Y' and decision == 'Exclude': #Duplication handling, also needs exclude decision
                     if full_text_decisions.get(article_id) != 'Exclude - Duplicate':
                         full_text_decisions[article_id] = 'Exclude - Duplicate'
                         with log_output:
                             print(f"Automatically set Full Text decision for duplicate Article {article_id} to Exclude - Duplicate.")
                 elif decision == 'Exclude': #Exclude based on title-abstract
                     full_text_decisions[article_id] = 'Exclude - Title-Abstract'
                     with log_output:
                         print(f"Automatically set Full Text decision for Article {article_id} to Exclude.")
                 elif decision == 'Exclude-Duplication': #Duplication identified
                     full_text_decisions[article_id] = 'Exclude - Duplicate'
                     with log_output:
                         print(f"Automatically set Full Text decision for Article {article_id} to Exclude.")
             else:
                 with log_output:
                     print("Error: No article is currently displayed to make a decision.")

        include_button.on_click(lambda b: make_decision('Include')) #Attaching the make_decision to the buttons
        exclude_button.on_click(lambda b: make_decision('Exclude'))
        duplication_button.on_click(lambda b: make_decision('Exclude-Duplication')) #User identified duplicate
        exclude_duplicate_button.on_click(lambda b: make_decision('Exclude')) #Atomaticly identified duplicate is also an 'Exclude' decision

        def update_article_display(_=None): # _=None to accept change argument from observe
             with log_output:
                 clear_output() #Clear only the log messages, not the article content or buttons

             current_displayed_article_info['article_id'] = None #Reset current article info

             try:
                 article_id = int(id_input.value)
                 current_font_size = font_size_slider.value

                 if 1 <= article_id <= len(df): #No out of bound
                     article = df.iloc[article_id - 1]
                     current_displayed_article_info['article_id'] = article_id
                     current_decision_label.value = f"Current Decision: {inclusion_decisions.get(article_id, 'Pending')}"

                     if article['DUPLICATION'] == 'Y':
                         title_display_widget.value = f"<p style='color:red;'>Article with ID {article_id} is a duplicate and its content will not be displayed.</p>"
                         abstract_display_widget.value = ""
                         buttons_container.children = (exclude_duplicate_button,)
                         return

                     article_title = article['TITLE']
                     article_abstract = article['ABSTRACT']

                     if article['DUPLICATION'] == 'S': #Mentioning if suspected duplicate
                        duplicate_of_id = article['DUPLICATE_OF_ID']
                        title_display_widget.value = (
                            f"<p><b>Title:</b> {article_title} "
                            f"<span style='color:darkorange; font-weight:bold;'>"
                            f"[Suspected duplicate of ID {int(duplicate_of_id)}]"
                            f"</span></p>")
                     else:
                        title_display_widget.value = f"<p><b>Title:</b> {article_title}</p>"

                     abstract_display_widget.value = f"<p><b>Abstract:</b></p><div style='white-space: pre-wrap; word-wrap: break-word; text-align: justify; font-size: {current_font_size}px; padding-right: 15px;'>{article_abstract}</div>"
                     buttons_container.children = (include_button, exclude_button, duplication_button)
                 else:
                     title_display_widget.value = ""
                     abstract_display_widget.value = f"Article with ID {article_id} is out of bounds. Please enter a valid ID between 1 and {len(df)}."
                     buttons_container.children = () #Clear buttons
                     current_decision_label.value = "Current Decision: Pending"
             except ValueError:
                 if id_input.value != '':
                     title_display_widget.value = ""
                     abstract_display_widget.value = "Invalid input. Please enter a number for Article ID."
                     buttons_container.children = () #Clear buttons
                     current_decision_label.value = "Current Decision: Pending"

        id_input.observe(update_article_display, names='value')
        font_size_slider.observe(update_article_display, names='value')
        update_article_display() #Initial display call
        display(widgets.VBox([
                id_input, font_size_slider,
                title_display_widget, abstract_display_widget,
                widgets.HTML("<br>"),
                buttons_container,
                current_decision_label, log_output
            ])) #Display all widgets


def full_text_inclusion(df, inclusion_decisions, full_text_decisions, full_text_exclusion_reasons, dynamic_output_area):
     with dynamic_output_area:
         clear_output(wait=True)
         id_input = widgets.Text(description="Article ID:", placeholder="Enter Article ID (e.g., 1, 2, ...)")
         title_display_widget = widgets.HTML(value="<p><b>Title:</b></p>")
         type_display_widget = widgets.HTML(value="<p><b>Type:</b></p>")
         doi_display_widget = widgets.HTML(value="<p><b>DOI:</b></p>")
         url_display_widget = widgets.HTML(value="<p><b>URL:</b></p>")
         include_button = widgets.Button(description="Include")
         exclude_button = widgets.Button(description="Exclude")
         current_decision_label = widgets.Label(value="Current Decision: Pending")

         buttons_container = widgets.HBox() #Include, Exclude button container
         exclusion_reason_buttons_container = widgets.HBox() #Container for dynamically generated exclusion reason buttons
         log_output = widgets.Output() #Output for messages/logs

         current_displayed_article_info = {'article_id': None}

         # --- Configuration Widgets for Exclusion Reasons ---
         current_reasons_label = widgets.HTML("<b>Current Exclusion Reasons:</b>")
         current_reasons_output = widgets.Output()
         new_reason_input = widgets.Text(description="Reason:")
         add_reason_button = widgets.Button(description="Add")
         remove_reason_button = widgets.Button(description="Remove")
         config_buttons_hbox = widgets.HBox([new_reason_input, add_reason_button, remove_reason_button])

         def update_current_reasons_display():
             with current_reasons_output:
                 clear_output()
                 display(widgets.HTML(f"<p>{', '.join(full_text_exclusion_reasons)}</p>"))

         def update_exclusion_reason_buttons():
             exclusion_reason_buttons_container.children = tuple(widgets.Button(description=reason) for reason in full_text_exclusion_reasons)
             for btn in exclusion_reason_buttons_container.children:
                 btn.on_click(lambda b_clicked: set_full_text_exclusion_reason(b_clicked.description))

         def on_add_reason_clicked(b):
             reason = new_reason_input.value.strip()
             if reason and reason not in full_text_exclusion_reasons:
                 full_text_exclusion_reasons.append(reason)
                 new_reason_input.value = ""
                 update_current_reasons_display()
                 with log_output:
                     print(f"Added exclusion reason: '{reason}'")
             elif reason in full_text_exclusion_reasons:
                 with log_output:
                     print(f"Reason '{reason}' already exists.")
             else:
                 with log_output:
                     print("Please enter a reason to add.")

         def on_remove_reason_clicked(b):
             reason = new_reason_input.value.strip()
             if reason and reason in full_text_exclusion_reasons:
                 full_text_exclusion_reasons.remove(reason)
                 new_reason_input.value = ""
                 update_current_reasons_display()
                 with log_output:
                     print(f"Removed exclusion reason: '{reason}'")
             elif reason not in full_text_exclusion_reasons:
                 with log_output:
                     print(f"Reason '{reason}' not found.")
             else:
                 with log_output:
                     print("Please enter a reason to remove.")

         add_reason_button.on_click(on_add_reason_clicked)
         remove_reason_button.on_click(on_remove_reason_clicked)
         # --- End Configuration Widgets ---

         def set_full_text_exclusion_reason(reason): #Callback for specific exclusion reason buttons
             nonlocal full_text_decisions
             article_id = current_displayed_article_info['article_id']

             if article_id is not None:
                 full_text_decisions[article_id] = f'Exclude - {reason}'
                 current_decision_label.value = f"Current Decision: {full_text_decisions.get(article_id, 'Pending')}"
                 buttons_container.children = (include_button, exclude_button) #Restore main buttons
                 exclusion_reason_buttons_container.children = () #Hide reason buttons
             else:
                 with log_output:
                     print("Error: No article is currently displayed to make a decision.")

         def make_decision(decision): #Callback for main Include/Exclude buttons
             nonlocal full_text_decisions
             article_id = current_displayed_article_info['article_id']

             if article_id is not None:
                 if decision == 'Include':
                     full_text_decisions[article_id] = 'Include'
                     current_decision_label.value = f"Current Decision: {full_text_decisions.get(article_id, 'Pending')}"
                     buttons_container.children = (include_button, exclude_button) #Ensure main buttons are shown if it was hidden for reason selection
                     exclusion_reason_buttons_container.children = () #Clear reason buttons if they were visible
                 elif decision == 'Exclude':
                     buttons_container.children = () #Hide Include/Exclude buttons
                     update_exclusion_reason_buttons() #Show dynamic reason buttons
             else:
                 with log_output:
                     print("Error: No article is currently displayed to make a decision.")

         include_button.on_click(lambda b: make_decision('Include')) #Attaching handlers
         exclude_button.on_click(lambda b: make_decision('Exclude'))

         def update_article_display(_=None):
             with log_output:
                 clear_output()

             current_displayed_article_info['article_id'] = None
             buttons_container.children = (include_button, exclude_button) #Always reset to initial button state when new article is loaded
             exclusion_reason_buttons_container.children = () #Hide dynamic exclusion reasons initially

             try:
                 article_id = int(id_input.value)

                 if 1 <= article_id <= len(df):
                     article = df.iloc[article_id - 1]
                     current_displayed_article_info['article_id'] = article_id
                     current_decision_label.value = f"Current Decision: {full_text_decisions.get(article_id, 'Pending')}"
                     title_abstract_decision = inclusion_decisions.get(article_id, 'Pending')
                     full_decision = full_text_decisions.get(article_id, 'Pending')

                     if full_decision == 'Exclude - Duplication':
                        title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""
                        url_display_widget.value = f"<p style='color:red;'>Article with ID {article_id} is a duplicate and its content will not be displayed. Full Text Include: N</p>"
                        buttons_container.children = () #No buttons if already excluded
                        exclusion_reason_buttons_container.children = ()
                        current_decision_label.value = f"Current Decision: {full_text_decisions.get(article_id, 'Pending')}"
                        return

                     if title_abstract_decision == 'Pending': #New condition: Check if Title-Abstract decision is pending
                         title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""
                         url_display_widget.value = f"<p style='color:red;'>Article with ID {article_id}: Title-Abstract screening is pending. Please complete Title-Abstract screening first.</p>"
                         buttons_container.children = () #No buttons if screening is pending
                         exclusion_reason_buttons_container.children = ()
                         current_decision_label.value = "Current Decision: Pending"
                         return

                     if title_abstract_decision == 'Exclude':
                         title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""
                         url_display_widget.value = f"<p style='color:red;'>Article with ID {article_id} was excluded during Title-Abstract screening. Full Text Include: N</p>"
                         buttons_container.children = () #No buttons if already excluded
                         exclusion_reason_buttons_container.children = ()
                         current_decision_label.value = f"Current Decision: {full_text_decisions.get(article_id, 'Pending')}"
                         return

                     if title_abstract_decision == 'Exclude-Duplication':
                        title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""
                        url_display_widget.value = f"<p style='color:red;'>Article with ID {article_id} is a duplicate and its content will not be displayed. Full Text Include: N</p>"
                        buttons_container.children = () #No buttons if already excluded
                        exclusion_reason_buttons_container.children = ()
                        current_decision_label.value = f"Current Decision: {full_text_decisions.get(article_id, 'Pending')}"
                        return

                     article_title = article['TITLE']
                     article_type = article['TYPE']
                     article_doi = article['DOI']
                     article_url = article['URL']
                     title_display_widget.value = f"<p><b>Title:</b> {article_title}</p>"
                     type_display_widget.value = f"<p><b>Type:</b> {article_type}</p>"

                     if article_doi and pd.notna(article_doi):
                         doi_display_widget.value = f"<p><b>DOI:</b> <a href='https://doi.org/{article_doi}' target='_blank'>{article_doi}</a></p>"
                     else:
                         doi_display_widget.value = "<p><b>DOI:</b> Not Available</p>"

                     if article_url and pd.notna(article_url):
                         url_display_widget.value = f"<p><b>URL:</b> <a href='{article_url}' target='_blank'>{article_url}</a></p>"
                     else:
                         url_display_widget.value = "<p><b>URL:</b> Not Available</p>"

                     buttons_container.children = (include_button, exclude_button) #Restore main buttons if they were hidden due to prior exclusion selection
                     exclusion_reason_buttons_container.children = ()

                 else:
                     title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""
                     url_display_widget.value = f"Artcile ID {article_id} is out of bounds. Please enter a valid ID between 1 and {len(df)}."
                     buttons_container.children = ()
                     exclusion_reason_buttons_container.children = ()
                     current_decision_label.value = "Current Decision: Pending"
             except ValueError:
                 if id_input.value != '':
                     title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""
                     url_display_widget.value = "Invalid input. Please enter a number for Article ID."
                     buttons_container.children = ()
                     exclusion_reason_buttons_container.children = ()
                     current_decision_label.value = "Current Decision: Pending"

         id_input.observe(update_article_display, names='value')
         update_current_reasons_display()
         update_article_display()
         display(widgets.VBox([
             current_reasons_label, current_reasons_output, config_buttons_hbox,
             widgets.HTML("<hr>"),
             widgets.HTML("<b>Full Text Screening</b>"),
             id_input,
             title_display_widget, type_display_widget, doi_display_widget, url_display_widget,
             widgets.HTML("<br>"),
             buttons_container, exclusion_reason_buttons_container,
             current_decision_label, log_output
        ]))

def data_extract_invalid_input(title_display_widget, doi_display_widget, url_display_widget, container, datas, data_text_widgets, message):
    title_display_widget.value = ""; doi_display_widget.value = ""
    url_display_widget.value = message
    container.layout.display = "none"
    for data_name in datas: #Clear data widgets if not included
           data_text_widgets[data_name].value = ""

# =============================================================================
# UI
# =============================================================================
############# DATABASE UPLOAD
def on_database_uploader_clicked(b_database):
    clear_output(wait=True) #Clearing the output
    display(HTML("<i>Database Upload</i>"))

    upload_button = widgets.Button(description="Uploader")
    viewer_button = widgets.Button(description="Viewer", disabled=True)
    dedup_button = widgets.Button(description="De-duplication", disabled=True)
    saving_button = widgets.Button(description="Save", disabled=True)
    dynamic_output_area = widgets.Output() #Output area for dynamically added widgets

    all_dfs = [] #DataFrames
    db_names = [] #Database names
    dedup_df_current = None
    dedup_stats = None
    inclusion_decisions = {} #For Title-Abstract decisions
    full_text_decisions = {} #For Full-Text decisions

    def on_upload_clicked(b_upload):
        with dynamic_output_area:
            clear_output(wait=True)
            db_label = widgets.HTML("<b>Database upload</b>")
            viewer_button.disabled, dedup_button.disabled, saving_button.disabled = True, True, True

            plus_button = widgets.Button(description="+ ")
            save_process_button = widgets.Button(description="Save and Process")
            save_process_button.disabled = True
            buttons_row = widgets.HBox([plus_button, save_process_button])
            db_entry_container = widgets.VBox([]) #Container for the database adder

            def on_plus_clicked(b_plus):
                name_input = widgets.Text(description="Name:")
                file_upload = widgets.FileUpload(description="Upload File", accept=".txt, .xml, .ris", multiple = False)
                add_button = widgets.Button(description="Add")
                added_label = widgets.Label(value="")

                def on_add_row_clicked(b_add_row):
                    if name_input.value:
                        if file_upload.value:
                            added_label.value = f"Ready to add '{name_input.value}'"
                            save_process_button.disabled = False
                        else:
                            added_label.value = "Please provide a file"
                    else:
                        added_label.value = "Please enter a name"
                add_button.on_click(on_add_row_clicked)
                new_entry_row = widgets.HBox([name_input, file_upload, add_button, added_label]) #Rows of the plus sign
                db_entry_container.children += (new_entry_row,) #Adding every row (database)

            def on_save_process_clicked(b_save):
                nonlocal all_dfs, db_names
                all_dfs.clear()
                db_names.clear()
                processed_count = 0

                for row_widgets in db_entry_container.children: #Iterating through every added rows of databases
                    name_input, file_upload, _, added_label = row_widgets.children
                    db_name = name_input.value
                    df_to_add = None

                    uploaded_file = list(file_upload.value.values())[0]
                    file_extension = uploaded_file['metadata']['name'].lower().split('.')[-1]
                    text = uploaded_file['content'].decode('utf-8')

                    try:
                        df_to_add = parse_text_data(text, file_extension) #Calling the text parser
                        if df_to_add is not None and not df_to_add.empty: #If the parsing was successful
                            all_dfs.append(df_to_add)
                            db_names.append(db_name)
                            added_label.value = f"Processed '{db_name}' from file."
                            processed_count += 1
                        else:
                            added_label.value = f"Error: No data processed from '{db_name}' file."
                    except Exception as e:
                        added_label.value = f"Error processing file for '{db_name}': {e}"

                with dynamic_output_area:
                    clear_output(wait=True)
                    if processed_count > 0:
                        display(HTML(f"<b>Processed {processed_count} database(s).</b>"))

                        try:
                            export_dataframes_to_excel(all_dfs, db_names, excel_output_file, header, prepare_worksheet) #Calling the excel exporter
                            display(HTML(f"<b>All DataFrames successfully saved to '{excel_output_file}'.</b>"))
                        except Exception as e:
                            display(HTML(f"<b style='color:red;'>Error saving to Excel: {e}</b>"))
                    else:
                        display(HTML("<b>No databases processed or saved.</b>"))

                    if len(all_dfs) > 0: #At least one correct database should be uploaded
                        dedup_button.disabled, viewer_button.disabled = False, False
                        upload_button.disabled = True
            plus_button.on_click(on_plus_clicked)
            save_process_button.on_click(on_save_process_clicked)
            display(widgets.VBox([db_label, buttons_row, db_entry_container]))

    def on_viewer_clicked(b_viewer):
        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>View DataFrames</b>"))
            all_db_rows = []

            for name, df in zip(db_names, all_dfs):
                out = widgets.Output()
                name_label = widgets.Label(value=name)
                viewhide_button = widgets.Button(description="View")

                def on_view_clicked(b, df=df, out=out):
                    with out:
                        clear_output()
                        if b.description == "View":
                            display(df)
                            b.description = "Hide"
                        else:
                            b.description = "View"
                viewhide_button.on_click(on_view_clicked)
                row = widgets.HBox([name_label, viewhide_button, out])
                all_db_rows.append(row)
            display(widgets.VBox(all_db_rows))

    def on_dedup_clicked(b_dedup):
      nonlocal dedup_df_current, dedup_stats
      with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>De-duplication Results</b>"))

            if dedup_df_current is None: #De-duplication was not performed before
                dedup_df_current, dedup_stats = deduplicate_all_dfs(all_dfs, db_names)

            #Creating a summary
            summary = [f"<i>Total records:</i> {dedup_stats['total_records']}"]

            for database_name, record_count in dedup_stats["records_by_database"].items():
                summary.append(f"<i>{database_name} records:</i> {record_count}")

            summary.append("-------------------------------",)
            summary.append(f"<i>Duplicated records:</i> {dedup_stats['duplicated_records']}")
            summary.append(f"<i>Suspected duplicated records:</i> {dedup_stats['suspected_duplicates']}")
            summary.append(f"<i>Non-duplicated records:</i> {dedup_stats['total_records']-dedup_stats['duplicated_records']}")

            summary_html = f"""<div style="display: inline-block; width: fit-content; max-width: 100%;
                border: 1px solid #ddd; padding: 10px; margin: 10px 0; border-radius: 6px;
                background-color: #f9f9f9; line-height: 1.6;">
                {"<br>".join(summary)}
            </div>"""

            display(HTML(summary_html))
            display(dedup_df_current)
            saving_button.disabled = False

    def on_saving_clicked(b_saving):
        nonlocal dedup_df_current, inclusion_decisions, full_text_decisions
        dedup_copy = dedup_df_current.copy() #To not append the dedup with screening columns
        saving_menu(dedup_copy, inclusion_decisions, full_text_decisions, dynamic_output_area, stage = "DB_Upload")

    upload_button.on_click(on_upload_clicked)
    viewer_button.on_click(on_viewer_clicked)
    dedup_button.on_click(on_dedup_clicked)
    saving_button.on_click(on_saving_clicked)
    display(widgets.HBox([upload_button, viewer_button, dedup_button, saving_button]))
    display(dynamic_output_area)

############# SCREENING
def on_screening_clicked(b_screening):
    clear_output(wait=True)
    display(HTML("<i>Screening</i>"))
    dynamic_output_area = widgets.Output()
    upload_button = widgets.Button(description="Uploader", disabled=False)
    title_abstract_button = widgets.Button(description="Title-Abstract", disabled=True)
    full_text_button = widgets.Button(description="Full-Text", disabled=True)
    final_button = widgets.Button(description="Final", disabled=True)

    continue_df = []
    uploaded_file_name = None
    inclusion_decisions = {}
    full_text_decisions = {}
    full_text_exclusion_reasons = ["No Full Text", "Not English", "Not Relevant"]

    def on_upload_clicked(b_upload):
        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Upload Existing Review File</b>"))
            file_upload = widgets.FileUpload(description="Upload Excel File", accept=".xlsx", multiple=False)
            add_button = widgets.Button(description="Add")
            save_process_button = widgets.Button(description="Load", disabled=True)
            added_label = widgets.Label(value="")

            def on_add_clicked(b_add):
                nonlocal uploaded_file_name
                if file_upload.value:
                    uploaded_file_name = list(file_upload.value.keys())[0]
                    added_label.value = f"Ready to load '{uploaded_file_name}'"
                    save_process_button.disabled = False
                else:
                    added_label.value = "Please select an Excel file."

            def on_save_process_clicked(b_save):
                nonlocal continue_df
                with dynamic_output_area:
                    clear_output(wait=True)

                    continue_df = excel_uploader(file_upload, uploaded_file_name)
                    if continue_df is None:
                        display(HTML(f"<b style='color:red;'>Error loading '{uploaded_file_name}'. There is no Final sheet in the uploaded file.</b>"))
                        return

                    for idx, row in continue_df.iterrows(): #Decision updating
                         article_id = row['ID']
                         ta_decision = (
                              "Include" if row['Title_Abstract_Include'] == "Y"
                               else "Exclude" if row['Title_Abstract_Include'] == "N"
                               else "Pending")

                         full_decision =  (
                               "Include" if row['Full_Text_Include'] == "Y"
                               else "Exclude - "+ str(row['Full_Text_Exclusion_Reason']) if row['Full_Text_Include'] == "N"
                               else "Pending")

                         inclusion_decisions[article_id] = ta_decision
                         full_text_decisions[article_id] = full_decision

                    display(HTML(f"<b>Loading from '{uploaded_file_name}'...</b>"))
                    title_abstract_button.disabled, full_text_button.disabled, final_button.disabled = False, False, False
                    upload_button.disabled = True
            add_button.on_click(on_add_clicked)
            save_process_button.on_click(on_save_process_clicked)
            display(widgets.VBox([file_upload, add_button, added_label, save_process_button]))

    def on_title_abstract_clicked(b_title_abstract):
        nonlocal continue_df, inclusion_decisions, full_text_decisions
        title_abstract_inclusion(continue_df, inclusion_decisions, full_text_decisions, dynamic_output_area)

    def on_full_text_clicked(b_full_text):
        nonlocal continue_df, inclusion_decisions, full_text_decisions
        full_text_inclusion(continue_df, inclusion_decisions, full_text_decisions, full_text_exclusion_reasons, dynamic_output_area)

    def on_final_clicked(b_continue):
        nonlocal inclusion_decisions, full_text_decisions, continue_df
        saving_menu(continue_df, inclusion_decisions, full_text_decisions, dynamic_output_area, stage = "Screening")

    upload_button.on_click(on_upload_clicked)
    title_abstract_button.on_click(on_title_abstract_clicked)
    full_text_button.on_click(on_full_text_clicked)
    final_button.on_click(on_final_clicked)
    display(widgets.HBox([upload_button, title_abstract_button, full_text_button, final_button]))
    display(dynamic_output_area)

############# COMPARE
def on_compare_clicked(b_compare):
    clear_output(wait=True)
    display(HTML("<i>Compare Decisions</i>"))
    dynamic_output_area = widgets.Output()
    upload_button = widgets.Button(description="Uploader", disabled=False)
    tiab_resolve_button = widgets.Button(description="Ti-Ab Compare", disabled=True)
    fullT_resolve_button = widgets.Button(description="Full-Text Compare", disabled=True)
    final_button = widgets.Button(description="Final", disabled=True)

    all_dfs = []
    final_df = None
    resolved_ids = []
    full_resolved_text = {}
    tiab_resolved_text = {}

    def on_upload_clicked(b_upload):
        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Upload Existing Review Files</b>"))
            plus_button = widgets.Button(description="+ ")
            save_process_button = widgets.Button(description="Save and Process")
            save_process_button.disabled = True
            buttons_row = widgets.HBox([plus_button, save_process_button])
            entry_container = widgets.VBox([])

            reviewer_count = 1

            def on_plus_clicked(b_plus):
                nonlocal reviewer_count
                name_label = widgets.Label(value=f"Reviewer {reviewer_count}:")
                reviewer_count += 1
                file_upload = widgets.FileUpload(description="Upload File", accept=".xlsx", multiple = False)
                add_button = widgets.Button(description="Add")
                added_label = widgets.Label(value="")

                def on_add_row_clicked(b_add_row):
                    if file_upload.value:
                        added_label.value = f"Ready to add"
                        if reviewer_count > 2: #Greater than 2 because the increment happens before
                            save_process_button.disabled = False
                    else:
                        added_label.value = "Please provide a file"
                add_button.on_click(on_add_row_clicked)
                new_entry_row = widgets.HBox([name_label, file_upload, add_button, added_label]) #Rows of the adder
                entry_container.children += (new_entry_row,) #Displaying every row

            def on_save_process_clicked(b_save):
                nonlocal all_dfs, final_df
                global excel_output_file
                with dynamic_output_area:
                    clear_output(wait=True)
                    all_dfs.clear()

                    for row_widgets in entry_container.children: #Iterating through every added rows of databases
                        name_label, file_upload, _, added_label = row_widgets.children

                        uploaded_file_name = list(file_upload.value.keys())[0]
                        df = excel_uploader(file_upload, uploaded_file_name)
                        if df is None:
                            display(HTML(f"<b style='color:red;'>Error loading '{uploaded_file_name}'. There is no Final sheet in the uploaded file.</b>"))
                            return
                        all_dfs.append(df)

                    with dynamic_output_area:
                        clear_output(wait=True)
                        display(HTML(f"<b>Processed.</b>"))
                        tiab_resolve_button.disabled, fullT_resolve_button.disabled, final_button.disabled = False, False, False
                        upload_button.disabled = True
                final_df = all_dfs[0].copy() #Saving the first file as a reference table for the Final sub-menu
                shutil.copyfile(excel_output_file, "Resolved_Review.xlsx")
                excel_output_file = "Resolved_Review.xlsx"
            plus_button.on_click(on_plus_clicked)
            save_process_button.on_click(on_save_process_clicked)
            display(widgets.VBox([buttons_row, entry_container]))

    def on_tiab_resolve_button_clicked(b_title_abstract):
        nonlocal all_dfs
        reviewer_count = 1

        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Title-Abstract Decision Comparison</b>"))
            title_display_widget = widgets.HTML(value="<p><b>Title:</b></p>")
            abstract_display_widget = widgets.HTML(value="<p><b>Abstract:</b></p>")
            count_display = widgets.HTML(value="")

            buttons_area = widgets.HBox() #Reviwer buttons
            navigation_buttons = widgets.HBox() #Previous and next buttons
            resolve_output = widgets.Output()
            log_output = widgets.Output()

            if any(df["Title_Abstract_Include"].eq("Pending").any() for df in all_dfs):
                display(HTML("<b style='color:red;'>The uploaded files contain articles with Pending Title-Abstract screening decisions.</b>"))
                return

            comparison_df = all_dfs[0][['ID', 'TITLE', 'ABSTRACT']].copy()
            reviewer_cols = []
            for i, df in enumerate(all_dfs): #Appending the decisions to the table
                col_name = f'Reviewer_{i+1}'
                reviewer_cols.append(col_name)

                reason = df["Full_Text_Exclusion_Reason"].fillna("").astype(str) #only compare tiab reasons
                comparison_df[col_name] = (df["Title_Abstract_Include"].fillna("").astype(str) + np.where(reason.isin(["Title-Abstract", "Duplicate"])," - " + reason, ""))

            disagreements = comparison_df[comparison_df[reviewer_cols].apply(lambda x: x.nunique() > 1, axis=1)].copy().reset_index(drop=True)
            disagreements["RESOLVED"] = disagreements["ID"].apply(lambda article_id: tiab_resolved_text.get(article_id, "NO")) #To not overwrite it every time
            current_index = [0]

            def update_article_display():
                buttons_area.children = []
                if disagreements.empty:
                    count_display.value = "<b>No disagreements found.</b>"
                    title_display_widget.value = ""; abstract_display_widget.value = ""
                    navigation_buttons.children = ()
                    resolve_output.clear_output()
                    log_output.clear_output()
                    return

                idx = current_index[0]
                article = disagreements.iloc[idx]
                article_id = article['ID']

                count_display.value = f"<b>{idx+1} of {len(disagreements)} disagreements</b>"
                title_display_widget.value = f"<p><b>Title:</b> {article['TITLE']}</p>"
                abstract_display_widget.value = f"<p><b>Abstract:</b></p><div style='white-space: pre-wrap;text-align: justify;padding-right: 15px;'>{article['ABSTRACT']}</div>"

                with resolve_output:
                    print(f"Resolved: {article['RESOLVED']}")

                reviewer_buttons = [] #Create decision buttons for each reviewer
                for i, df in enumerate(all_dfs):
                    reviewer_data = df[df['ID'] == article_id].iloc[0]
                    desc = f"Reviewer {i+1}: {reviewer_data['Title_Abstract_Include']} | Reason - {reviewer_data['Full_Text_Exclusion_Reason']}"
                    btn = widgets.Button(description=desc, layout=widgets.Layout(width='auto'))

                    def on_decision_selected(b, r_data=reviewer_data):
                        nonlocal final_df, resolved_ids
                        mask = final_df['ID'] == article_id #Update final_df for this article
                        final_df.loc[mask, 'Title_Abstract_Include'] = r_data['Title_Abstract_Include']
                        final_df.loc[mask, 'Full_Text_Include'] = r_data['Full_Text_Include']
                        final_df.loc[mask, 'Full_Text_Exclusion_Reason'] = r_data['Full_Text_Exclusion_Reason']

                        resolved_ids.append(article_id)
                        resolved_text = f"YES - (Decision: {r_data['Title_Abstract_Include']} | Exclusion-Reason: {r_data['Full_Text_Exclusion_Reason']})"
                        tiab_resolved_text[article_id] = resolved_text
                        disagreements.loc[idx, "RESOLVED"] = resolved_text

                        resolve_output.clear_output()
                        with log_output:
                            print(f"ID {article_id} updated.")
                    btn.on_click(on_decision_selected)
                    reviewer_buttons.append(btn)
                buttons_area.children = reviewer_buttons

            prev_btn = widgets.Button(description="Previous")
            next_btn = widgets.Button(description="Next")
            navigation_buttons = widgets.HBox([prev_btn, next_btn])

            def on_prev(_):
                resolve_output.clear_output()
                log_output.clear_output()

                if current_index[0] > 0:
                    current_index[0] -= 1
                    update_article_display()
                else:
                    update_article_display()

            def on_next(_):
                resolve_output.clear_output()
                log_output.clear_output()

                if current_index[0] < len(disagreements) - 1:
                    current_index[0] += 1
                    update_article_display()
                else:
                    update_article_display()
            prev_btn.on_click(on_prev)
            next_btn.on_click(on_next)

            update_article_display()
            display(widgets.VBox([count_display,
                navigation_buttons,
                title_display_widget, abstract_display_widget,
                buttons_area,
                widgets.HTML("<br>"),
                resolve_output, log_output
            ]))

    def on_fullT_resolve_button_clicked(b_fullT):
        nonlocal all_dfs, full_resolved_text
        reviewer_count = 1

        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Full-Text Decision Comparison</b>"))
            title_display_widget = widgets.HTML(value="<p><b>Title:</b></p>")
            type_display_widget = widgets.HTML(value="<p><b>Type:</b></p>")
            doi_display_widget = widgets.HTML(value="<p><b>DOI:</b></p>")
            url_display_widget = widgets.HTML(value="<p><b>URL:</b></p>")
            count_display = widgets.HTML(value="")
            buttons_area = widgets.HBox() #Reviwer buttons
            navigation_buttons = widgets.HBox() #Previous and next buttons
            resolve_output = widgets.Output()
            log_output = widgets.Output()

            if any(df["Full_Text_Include"].eq("Pending").any() for df in all_dfs):
                display(HTML("<b style='color:red;'>The uploaded files contain articles with Pending Full-Text screening decisions.</b>"))
                return

            comparison_df = all_dfs[0][['ID', 'TITLE', 'TYPE', 'DOI', 'URL']].copy()
            reviewer_cols = []
            for i, df in enumerate(all_dfs): #Appending the decisions to the table
                col_name = f'Reviewer_{i+1}'
                reviewer_cols.append(col_name)
                comparison_df[col_name] = df['Full_Text_Include'].fillna(" ").astype(str) + " - " + df['Full_Text_Exclusion_Reason'].fillna(" ").astype(str)

            disagreements = comparison_df[comparison_df[reviewer_cols].apply(lambda x: x.nunique() > 1, axis=1)].copy().reset_index(drop=True)
            disagreements["RESOLVED"] = disagreements["ID"].apply(lambda article_id: full_resolved_text.get(article_id, "NO")) #To not overwrite it every time
            current_index = [0]

            def update_article_display():
                buttons_area.children = []
                if disagreements.empty:
                    count_display.value = "<b>No disagreements found.</b>"
                    title_display_widget.value = ""; type_display_widget.value = ""; doi_display_widget.value = ""; url_display_widget.value = ""
                    navigation_buttons.children = ()
                    resolve_output.clear_output()
                    log_output.clear_output()
                    return

                idx = current_index[0]
                article = disagreements.iloc[idx]
                article_id = article['ID']

                count_display.value = f"<b>{idx+1} of {len(disagreements)} disagreements</b>"
                title_display_widget.value = f"<p><b>Title:</b> {article['TITLE']}</p>"
                type_display_widget.value = f"<p><b>Type:</b> {article['TYPE']}</p>"

                doi = str(article.get('DOI', ''))
                doi_link = f"<a href='https://doi.org/{doi}' target='_blank'>{doi}</a>" if doi and doi != 'nan' else 'N/A'
                doi_display_widget.value = f"<p><b>DOI:</b> {doi_link}</p>"

                url = str(article.get('URL', ''))
                url_link = f"<a href='{url}' target='_blank'>{url}</a>" if url and url != 'nan' else 'N/A'
                url_display_widget.value = f"<p><b>URL:</b> {url_link}</p>"

                with resolve_output:
                    print(f"Resolved: {article['RESOLVED']}")

                reviewer_buttons = [] #Create decision buttons for each reviewer
                for i, df in enumerate(all_dfs):
                    reviewer_data = df[df['ID'] == article_id].iloc[0]
                    desc = f"Reviewer {i+1}: {reviewer_data['Full_Text_Include']} | Reason - {reviewer_data['Full_Text_Exclusion_Reason']}"
                    btn = widgets.Button(description=desc, layout=widgets.Layout(width='auto'))

                    def on_decision_selected(b, r_data=reviewer_data):
                        nonlocal final_df, resolved_ids
                        mask = final_df['ID'] == article_id #Update final_df for this article
                        final_df.loc[mask, 'Full_Text_Include'] = r_data['Full_Text_Include']
                        final_df.loc[mask, 'Full_Text_Exclusion_Reason'] = r_data['Full_Text_Exclusion_Reason']

                        resolved_ids.append(article_id)

                        resolved_text = f"YES - (Decision: {r_data['Full_Text_Include']} | Exclusion-Reason: {r_data['Full_Text_Exclusion_Reason']})"
                        full_resolved_text[article_id] = resolved_text
                        disagreements.loc[idx, "RESOLVED"] = resolved_text

                        resolve_output.clear_output()
                        with log_output:
                            print(f"ID {article_id} updated.")
                    btn.on_click(on_decision_selected)
                    reviewer_buttons.append(btn)
                buttons_area.children = reviewer_buttons

            prev_btn = widgets.Button(description="Previous")
            next_btn = widgets.Button(description="Next")
            navigation_buttons = widgets.HBox([prev_btn, next_btn])

            def on_prev(_):
                resolve_output.clear_output()
                log_output.clear_output()

                if current_index[0] > 0:
                    current_index[0] -= 1
                    update_article_display()
                else:
                    update_article_display()

            def on_next(_):
                resolve_output.clear_output()
                log_output.clear_output()

                if current_index[0] < len(disagreements) - 1:
                    current_index[0] += 1
                    update_article_display()
                else:
                    update_article_display()
            prev_btn.on_click(on_prev)
            next_btn.on_click(on_next)

            update_article_display()
            display(widgets.VBox([count_display,
                navigation_buttons,
                title_display_widget, type_display_widget, doi_display_widget, url_display_widget,
                buttons_area,
                widgets.HTML("<br>"),
                resolve_output, log_output
            ]))

    def on_final_clicked(b_continue):
        nonlocal final_df
        saving_menu(final_df, None, None, dynamic_output_area, stage = "Comparision")

    upload_button.on_click(on_upload_clicked)
    tiab_resolve_button.on_click(on_tiab_resolve_button_clicked)
    fullT_resolve_button.on_click(on_fullT_resolve_button_clicked)
    final_button.on_click(on_final_clicked)
    display(widgets.HBox([upload_button, tiab_resolve_button, fullT_resolve_button, final_button]))
    display(dynamic_output_area)

######## DATA EXTRACTION
def on_extraction_clicked(b_compare):
    clear_output(wait=True)
    display(HTML("<i>Data Extraction</i>"))
    dynamic_output_area = widgets.Output()
    upload_button = widgets.Button(description="Uploader", disabled=False)
    configuration_button = widgets.Button(description="Configuration", disabled=True)
    data_extraction_button = widgets.Button(description="Extraction", disabled=True)
    final_button = widgets.Button(description="Final", disabled=True)

    datas = []
    final_df = None
    uploaded_file_name = None
    full_text_exclusion_reasons = ["No Full Text", "Not English", "Not Relevant"]
    conf_check = False

    def on_upload_clicked(b_upload):
        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Upload Existing Review File</b>"))
            file_upload = widgets.FileUpload(description="Upload Excel File", accept=".xlsx", multiple=False)
            add_button = widgets.Button(description="Add")
            save_process_button = widgets.Button(description="Load", disabled=True)
            added_label = widgets.Label(value="")

            def on_add_clicked(b_add):
                nonlocal uploaded_file_name
                if file_upload.value:
                    uploaded_file_name = list(file_upload.value.keys())[0]
                    added_label.value = f"Ready to load '{uploaded_file_name}'"
                    save_process_button.disabled = False
                else:
                    added_label.value = "Please select an Excel file."

            def on_save_process_clicked(b_save):
                nonlocal final_df, conf_check
                with dynamic_output_area:
                    clear_output(wait=True)

                    final_df = excel_uploader(file_upload, uploaded_file_name)
                    if final_df is None:
                        display(HTML(f"<b style='color:red;'>Error loading '{uploaded_file_name}'. There is no Final sheet in the uploaded file.</b>"))
                        return

                    check_columns = ["Title_Abstract_Include", "Full_Text_Include"] #Check if every article has a screening decision

                    if final_df[check_columns].eq("Pending").any().any():
                        display(HTML(f"<b style='color:red;'>The uploaded file contains articles with Pending screening decisions.</b>"))
                        return

                    display(HTML(f"<b>Loading from '{uploaded_file_name}'...</b>")) #If there is no problem tell the user that the file is loading

                    if final_df.columns[-1] != "Full_Text_Exclusion_Reason" and conf_check == False: #Check if the configuration was done before
                        datas.extend(list(final_df.columns[final_df.columns.get_loc("Full_Text_Exclusion_Reason") + 1:]))
                        conf_check = True
                        display(HTML(f"<b>Loading the previously saved configuration and associated data...</b>"))

                    configuration_button.disabled, data_extraction_button.disabled, final_button.disabled = False, False, False
                    upload_button.disabled = True
            add_button.on_click(on_add_clicked)
            save_process_button.on_click(on_save_process_clicked)
            display(widgets.VBox([file_upload, add_button, added_label, save_process_button]))

    def on_configuration_clicked(b_configuration):
        nonlocal datas, final_df, conf_check
        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Extraction Configuration</b>"))
            plus_button = widgets.Button(description="+ ")
            save_process_button = widgets.Button(description="Save and Process")
            save_process_button.disabled = True
            buttons_row = widgets.HBox([plus_button, save_process_button])
            entry_container = widgets.VBox([])

            if datas:
                display(HTML(f"<br>Previously configured aspect(s): {datas}"))

            def on_plus_clicked(b_plus):
                data = widgets.Text(description="Data:")
                add_button = widgets.Button(description="Add")
                added_label = widgets.Label(value="")

                def on_add_row_clicked(b_add_row):
                    if data.value and data.value not in datas:
                        added_label.value = f"Ready to add"
                        save_process_button.disabled = False
                    else:
                        message = f"Aspect '{data.value}' already exists."
                        if added_label.value != message:
                            added_label.value = message
                        save_process_button.disabled = True
                add_button.on_click(on_add_row_clicked)
                new_entry_row = widgets.HBox([data, add_button, added_label]) #Rows of the adder
                entry_container.children += (new_entry_row,) #Displaying every row

            def on_save_process_clicked(b_save):
                for row_widgets in entry_container.children: #Iterating through every added rows of databases
                    data, _, added_label = row_widgets.children
                    datas.append(data.value)

                for col_name in datas: #Add new columns
                    if col_name not in final_df.columns:
                        final_df[col_name] = ""

                with dynamic_output_area:
                    clear_output(wait=True)
                    display(HTML(f"<b>Processed.</b>"))
                    print(f"Datas: {datas}")
            plus_button.on_click(on_plus_clicked)
            save_process_button.on_click(on_save_process_clicked)
            display(widgets.VBox([buttons_row, entry_container]))

    def on_data_extraction_clicked(b_data_extraction):
        nonlocal datas, final_df
        with dynamic_output_area:
            clear_output(wait=True)
            display(HTML("<b>Extraction</b>"))
            row_id_input = widgets.Text(description="Article ID:", placeholder="Enter Article ID (e.g., 1, 2, ...)")
            title_display_widget = widgets.HTML(value="<p><b>Title:</b></p>")
            doi_display_widget = widgets.HTML(value="<p><b>DOI:</b></p>")
            url_display_widget = widgets.HTML(value="<p><b>URL:</b></p>")

            data_text_widgets = {}
            data_rows = []

            for data_name in datas:
                label = widgets.Label(value=str(data_name), layout=widgets.Layout(width="100px"))
                text_widget = widgets.Text(value="", layout=widgets.Layout(width="600px"))
                data_text_widgets[data_name] = text_widget
                data_rows.append(widgets.HBox([label, text_widget]))

            container = widgets.VBox(data_rows)
            log_output = widgets.Output()
            save_data_button = widgets.Button(description="Save Data")

            included_df = final_df[(final_df['Title_Abstract_Include'] == 'Y') & (final_df['Full_Text_Include'] == 'Y')]
            count_display = widgets.HTML(value=f"{len(included_df)} articles were included from {len(final_df)}.")

            def on_save_data_clicked(b):
                nonlocal final_df
                with log_output:
                    clear_output()
                    try:
                        current_article_id = int(row_id_input.value)

                        if current_article_id and current_article_id in final_df['ID'].values:
                            for data_name in datas:
                                final_df.loc[final_df['ID'] == current_article_id, data_name] = data_text_widgets[data_name].value
                            print(f"Data for Article ID {current_article_id} saved.")
                        else:
                            print("No article selected or invalid article ID to save data for.")
                    except ValueError:
                        print("Invalid Article ID. Please enter a valid number.")
            save_data_button.on_click(on_save_data_clicked)

            #ID based article update
            def update_article_display(_=None):
                nonlocal container
                with log_output:
                    clear_output()
                #prev_btn.disabled, next_btn.disabled, save_data_button.disabled = True, True, True
                save_data_button.disabled = True

                try:
                    row_id = int(row_id_input.value)

                    if 1 <= row_id <= len(final_df):
                        article = final_df.iloc[row_id - 1]

                        if article['ID'] in included_df['ID'].values:
                            container.layout.display = "flex"

                            title_display_widget.value = f"<p><b>Title:</b> {article['TITLE']}</p>"

                            doi = str(article.get('DOI', ''))
                            doi_link = f"<a href='https://doi.org/{doi}' target='_blank'>{doi}</a>" if doi and doi != 'nan' else 'N/A'
                            doi_display_widget.value = f"<p><b>DOI:</b> {doi_link}</p>"

                            url = str(article.get('URL', ''))
                            url_link = f"<a href='{url}' target='_blank'>{url}</a>" if url and url != 'nan' else 'N/A'
                            url_display_widget.value = f"<p><b>URL:</b> {url_link}</p>"

                            for data_name in datas: #Populate data_text_widgets with existing data
                                data_text_widgets[data_name].value = str(article.get(data_name, ""))
                            save_data_button.disabled = False #Enable save button
                        else:
                            data_extract_invalid_input(title_display_widget, doi_display_widget, url_display_widget, container, datas, data_text_widgets, message = f"Artcile ID {article['ID']} is was not included during screening.")
                    else:
                        data_extract_invalid_input(title_display_widget, doi_display_widget, url_display_widget, container, datas, data_text_widgets, message = f"Artcile ID {row_id} is out of bounds. Please enter a valid ID between 1 and {len(final_df)}.")
                except ValueError:
                    data_extract_invalid_input(title_display_widget, doi_display_widget, url_display_widget, container, datas, data_text_widgets, message = "Invalid input. Please enter a number for Article ID.")

            row_id_input.observe(update_article_display, names='value')

            #Button based article update
            first_btn = widgets.Button(description="First")
            prev_btn = widgets.Button(description="Previous")
            next_btn = widgets.Button(description="Next")

            def show_included_article(position):
                position = max(0, min(position, len(included_df) - 1)) #Index always between 0 and the len
                article_id = included_df.iloc[position]["ID"]
                row_id_input.value = str(article_id) #Will trigger update_article_display

            def on_first(_):
                show_included_article(0)

            def on_prev(_):
                entered_id = row_id_input.value
                if entered_id is "": #If there is no id entered show the first included
                    show_included_article(0)
                    return

                entered_id = int(entered_id)
                included_ids = included_df["ID"].to_numpy()
                position = included_ids.searchsorted(entered_id, side="left")

                previous_position = max(0, position - 1) #If there is no previous, display the first one
                show_included_article(previous_position)

            def on_next(_):
                entered_id = row_id_input.value
                if entered_id is "": #If there is no id entered show the first included
                    show_included_article(0)
                    return

                entered_id = int(entered_id)
                included_ids = included_df["ID"].to_numpy()
                next_position = included_ids.searchsorted(entered_id, side="right")

                next_position = min(next_position, len(included_ids) - 1) #If there is no next, display the last one
                show_included_article(next_position)

            first_btn.on_click(on_first)
            prev_btn.on_click(on_prev)
            next_btn.on_click(on_next)

            #Main display
            display(widgets.VBox([
                row_id_input,
                widgets.HBox([first_btn, prev_btn, next_btn]),
                title_display_widget, doi_display_widget, url_display_widget,
                widgets.HTML("<br>"),
                container,
                widgets.HBox([save_data_button]),
                log_output, count_display
            ])) #Display all widgets

    def on_final_clicked(b_continue):
        nonlocal final_df
        saving_menu(final_df, None, None, dynamic_output_area, stage = "Extraction")

    upload_button.on_click(on_upload_clicked)
    configuration_button.on_click(on_configuration_clicked)
    data_extraction_button.on_click(on_data_extraction_clicked)
    final_button.on_click(on_final_clicked)
    display(widgets.HBox([upload_button, configuration_button, data_extraction_button, final_button]))
    display(dynamic_output_area)

display_main_menu() #Initial call to display the main menu